# Diagnosticus — versão ordinal com 4 thresholds fixos

Mudanças mínimas em relação ao CredituS binário:
1. **Carregamento** — usa `iloc[:, 1:-2]` (a base tem coluna extra ao final) e mapeia rótulos textuais para inteiros 0–4.
2. **`calcular_fitness`** — troca `np.where(q>=0, 1, 0)` por `np.digitize(q, THRESHOLDS)` e o produto `pa*pi` por um loop de produto sobre K classes.
3. Resto do arquivo — `criar_cromossomos`, `fitness_percentual`, `selecionar_pais_roleta`, `cruzar_pais`, `mutar`, `atualizar_populacao`, `algoritmo_genetico` — **não muda uma linha**.

In [11]:
import numpy as np
import pandas as pd

# ─── MUDANÇA 1: mapa de rótulos e slicing correto para esta base ───
mapa = {'<=-3': 0, '-3<g<=-1': 1, '-1<s<=1': 2, '1<p<=3': 3, '>3': 4}

df = pd.read_excel("BancoDiagnosticus.xlsx")
df = df[df['R'].isin(mapa)]              # remove 6 linhas com gabarito ruim

dataframe_dados_clientes = df.iloc[:, 1:-2]   # antes: 1:-1 (essa base tem coluna extra no fim)
dataframe_gabarito       = df['R'].map(mapa)  # antes: df.iloc[:, -1]

array_dados_clientes = dataframe_dados_clientes.values.astype(float)
array_gabarito       = dataframe_gabarito.values

qtd_features = array_dados_clientes.shape[1]
qtd_genes    = qtd_features + 1               # features + bias

print(f"Base carregada: {array_dados_clientes.shape[0]} clientes, {qtd_features} features")

Base carregada: 393 clientes, 30 features


In [12]:
# ─── MUDANÇA 2: thresholds do domínio + fitness em produto sobre K classes ───
THRESHOLDS = np.array([-3, -1, 1, 3])          # cortes dados pelo problema
K = len(THRESHOLDS) + 1                        # 5 classes ordinais


def criar_cromossomos(qtd_cromossomos: int = 6, qtd_genes: int = 19) -> np.ndarray:
    return -1 + 2 * np.random.rand(qtd_cromossomos, qtd_genes)


def calcular_fitness(cromossomos: np.ndarray, array_dados_clientes: np.ndarray, array_gabarito: np.ndarray) -> np.ndarray:
    """
    Ordinal com K classes e (K-1) thresholds FIXOS do domínio.
    Projeta q = X·w + bias, classifica por np.digitize contra THRESHOLDS,
    e o fitness é o produto dos recalls por classe (Prandiano generalizado).
    """
    totais = np.array([np.sum(array_gabarito == k) for k in range(K)])

    lista_fitness = []
    for linha in cromossomos:
        bias  = linha[0]
        genes = linha[1:]

        q = np.dot(array_dados_clientes, genes) + bias
        vetor_hipotese = np.digitize(q, THRESHOLDS)   # antes: np.where(q>=0, 1, 0)

        fitness = 1.0
        for k in range(K):                            # antes: pa * pi direto
            acertos_k = np.sum((vetor_hipotese == k) & (array_gabarito == k))
            fitness  *= acertos_k / totais[k]

        lista_fitness.append(fitness)

    return np.array(lista_fitness)


# ─── daqui pra baixo: idêntico ao original ───
def fitness_percentual(vetor_fitnesses: np.ndarray) -> np.ndarray:
    soma = np.sum(vetor_fitnesses)
    if soma == 0:
        return np.ones(len(vetor_fitnesses)) / len(vetor_fitnesses)
    return vetor_fitnesses / soma


def selecionar_pais_roleta(cromossomos: np.ndarray, percentual_fitnesses: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    roleta_acumulada = np.cumsum(percentual_fitnesses)
    indice_pai = min(int(np.searchsorted(roleta_acumulada, np.random.rand())), len(cromossomos) - 1)
    indice_mae = min(int(np.searchsorted(roleta_acumulada, np.random.rand())), len(cromossomos) - 1)
    return cromossomos[indice_pai], cromossomos[indice_mae]


def cruzar_pais(pai: np.ndarray, mae: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    c1 = np.random.randint(1, len(pai))
    c2 = np.random.randint(1, len(pai))
    c3 = np.random.randint(1, len(pai))

    filho1 = np.concatenate([pai[:c1], mae[c1:]])
    filho2 = np.concatenate([pai[:c2], mae[c2:]])
    filho3 = np.concatenate([pai[:c3], mae[c3:]])
    return filho1, filho2, filho3


def mutar(filho1: np.ndarray, filho2: np.ndarray, filho3: np.ndarray, qtd_genes_mutados: int = 1) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    for filho in [filho1, filho2, filho3]:
        k = min(qtd_genes_mutados, len(filho))
        indices = np.random.choice(len(filho), size=k, replace=False)
        filho[indices] = -1 + 2 * np.random.rand(k)
    return filho1, filho2, filho3


def atualizar_populacao(cromossomos: np.ndarray, vetor_fitnesses: np.ndarray,
                        filho1: np.ndarray, filho2: np.ndarray, filho3: np.ndarray,
                        array_dados_clientes: np.ndarray, array_gabarito: np.ndarray) -> np.ndarray:
    filhos = np.array([filho1, filho2, filho3])
    fitnesses_filhos = calcular_fitness(filhos, array_dados_clientes, array_gabarito)

    indices_melhores_filhos = np.argsort(fitnesses_filhos)[-2:]
    indices_piores          = np.argsort(vetor_fitnesses)[:2]

    nova_populacao = cromossomos.copy()
    for i in range(2):
        idx_pior  = int(indices_piores[i])
        idx_filho = int(indices_melhores_filhos[i])
        nova_populacao[idx_pior] = filhos[idx_filho]

    return nova_populacao

In [13]:
def algoritmo_genetico(
    array_dados_clientes: np.ndarray,
    array_gabarito: np.ndarray,
    qtd_cromossomos: int = 6,
    qtd_genes: int = 19,
    qtd_genes_mutados: int = 1,
    geracoes: int = 100,
    fitness_alvo: float = 0.90,
) -> tuple[np.ndarray, float]:

    populacao = criar_cromossomos(qtd_cromossomos, qtd_genes)

    melhor_cromossomo = None
    melhor_fitness = 0.0

    for geracao in range(geracoes):
        fitnesses = calcular_fitness(populacao, array_dados_clientes, array_gabarito)
        percentuais = fitness_percentual(fitnesses)

        idx_melhor = int(np.argmax(fitnesses))
        fitness_atual = float(fitnesses[idx_melhor])

        if fitness_atual > melhor_fitness:
            melhor_fitness = fitness_atual
            melhor_cromossomo = populacao[idx_melhor].copy()

        print(f"Geração {geracao+1:>4} | melhor fitness: {melhor_fitness:.4f}")

        if melhor_fitness >= fitness_alvo:
            print(f"\nFitness alvo {fitness_alvo} atingido na geração {geracao+1}.")
            break

        pai, mae = selecionar_pais_roleta(populacao, percentuais)
        filho1, filho2, filho3 = cruzar_pais(pai, mae)
        filho1, filho2, filho3 = mutar(filho1, filho2, filho3, qtd_genes_mutados)
        populacao = atualizar_populacao(
            populacao, fitnesses,
            filho1, filho2, filho3,
            array_dados_clientes, array_gabarito,
        )

    return melhor_cromossomo, melhor_fitness

### Rodar

Ajustes em relação ao CredituS:
- `qtd_genes=qtd_genes` (31 pra essa base, calculado automaticamente do shape)
- `fitness_alvo=0.60` — produto de 5 recalls comprime muito mais que produto de 2. Fitness 0.60 corresponde a ~90% médio por classe (G-mean = 0.60^(1/5) ≈ 0.903).

In [14]:
resultado = algoritmo_genetico(
    array_dados_clientes,
    array_gabarito,
    qtd_cromossomos=100,
    qtd_genes=qtd_genes,      
    qtd_genes_mutados=10,
    geracoes=3000,
    fitness_alvo=0.90,
)

melhor_cromossomo, melhor_fitness = resultado

Geração    1 | melhor fitness: 0.0000
Geração    2 | melhor fitness: 0.0000
Geração    3 | melhor fitness: 0.0000
Geração    4 | melhor fitness: 0.0000
Geração    5 | melhor fitness: 0.0000
Geração    6 | melhor fitness: 0.0000
Geração    7 | melhor fitness: 0.0017
Geração    8 | melhor fitness: 0.0017
Geração    9 | melhor fitness: 0.0022
Geração   10 | melhor fitness: 0.0022
Geração   11 | melhor fitness: 0.0054
Geração   12 | melhor fitness: 0.0054
Geração   13 | melhor fitness: 0.0054
Geração   14 | melhor fitness: 0.0054
Geração   15 | melhor fitness: 0.0054
Geração   16 | melhor fitness: 0.0054
Geração   17 | melhor fitness: 0.0054
Geração   18 | melhor fitness: 0.0122
Geração   19 | melhor fitness: 0.0122
Geração   20 | melhor fitness: 0.0122
Geração   21 | melhor fitness: 0.0122
Geração   22 | melhor fitness: 0.0122
Geração   23 | melhor fitness: 0.0122
Geração   24 | melhor fitness: 0.0122
Geração   25 | melhor fitness: 0.0122
Geração   26 | melhor fitness: 0.0122
Geração   27

### Diagnóstico da solução

In [15]:
bias, genes = melhor_cromossomo[0], melhor_cromossomo[1:]
q = array_dados_clientes @ genes + bias
h = np.digitize(q, THRESHOLDS)

acc = np.mean(h == array_gabarito)
gmean = melhor_fitness ** (1/K) if melhor_fitness > 0 else 0

print(f"Fitness (produto de {K} recalls): {melhor_fitness:.4f}")
print(f"G-mean (fitness^(1/{K})):         {gmean:.4f}")
print(f"Acurácia bruta:                  {acc:.4f}")
print(f"\nMatriz de confusão {K}x{K}:")
print("       Prev:  C0   C1   C2   C3   C4")
for k in range(K):
    linha = [int(np.sum((array_gabarito == k) & (h == j))) for j in range(K)]
    tot = sum(linha)
    rec = linha[k] / tot if tot else 0
    print(f"  Real C{k}: " + " ".join(f"{v:4d}" for v in linha) + f"    recall={rec:.3f}")

Fitness (produto de 5 recalls): 0.1087
G-mean (fitness^(1/5)):         0.6416
Acurácia bruta:                  0.6183

Matriz de confusão 5x5:
       Prev:  C0   C1   C2   C3   C4
  Real C0:   36    4    0    0    0    recall=0.900
  Real C1:   44   58   18    0    0    recall=0.483
  Real C2:    2   10   61    5    1    recall=0.772
  Real C3:    4    5   23   49    1    recall=0.598
  Real C4:    1    1    9   22   39    recall=0.542
